# IAC 2025 replication — TODO

Replication of Hottenrott & Lawson (2021), *What's behind multiple institutional affiliations in academia?* on the 2025 IAC survey data.

## Phase 1 — Baseline replication

### 1.1 Data prep
- [x] Load `all2025.dta` and sanity-check row counts per country
- [x] Recode A1 → 3-level seniority (junior / mid / senior)
- [ ] Drop non-academic respondents (A1 = non-academic / emeritus / not employed)
- [ ] Build binary + 4-level categorical versions of the 12 B12 motivation items
- [ ] Impute missing motivations by seniority × country × field cell means (for multiaffil==1)
- [ ] Build `satisfaction` index from E2 (research-relevant items, mean imputation)
- [ ] Coalesce B6 + B6P → unified 5 purpose binaries
- [ ] Coalesce B8 + B8P → unified 6 origin binaries
- [ ] Apply affiliation window filter (last ~5 years via B4 / B4P)
- [ ] Source external data: publications / citations (Scopus)
- [ ] Source external data: institution rank (THE + national, 4 tiers)
- [ ] Source external data: PRO classification
- [ ] Source external data: scientific field (sampling frame)

### 1.2 Descriptive reproduction
- [ ] Overall co-affiliation rate (paper: 25.7% → new: ?)
- [ ] Table 2 equivalent: t-tests by co-affiliation status
- [ ] Figure 2 equivalent: motivation item distributions
- [ ] Publication-listing stats (B10)

### 1.3 Factor analysis
- [ ] PCA with varimax on 12 B12 items → reproduce Table B.1
- [ ] Factor loadings → reproduce Table B.2
- [ ] Validate 4-factor structure (Network/Prestige, Resources, Teaching, Income)

### 1.4 K-means clustering
- [ ] KModes (simple matching) on 10 variables (purpose + origin, excl. personal contacts), k=4, seed=12
- [ ] Reproduce Table 3 cluster means
- [ ] Calinski-Harabasz sensitivity across 200 seeds

### 1.5 Two-stage probit selection
- [ ] Bivariate probit with selection (Python equivalent of `cmp ind(recentma*4 4)`)
- [ ] Stage 1: P(multiaffil) on full sample
- [ ] Stage 2: P(cluster Qk) for Q1/Q2/Q3/Q4
- [ ] Reproduce Table 5 (marginal effects)

### 1.6 Robustness
- [ ] Alternative clustering from heuristic seeds → reproduce Table 7
- [ ] Heckprob and manual Mills ratio as validation

## Phase 2 — Temporal comparison
- [ ] Country-level changes in co-affiliation rate vs. 2016
- [ ] Does the 4-cluster typology still fit?
- [ ] Have motivation-to-cluster mappings shifted?

## Phase 3 — Extensions (new survey sections)
- [ ] Section C analyses: data sharing practices × co-affiliation status/type
- [ ] Section D analyses: AI use × co-affiliation status/type

## Phase 4 — Validation (if 2016 data available)
- [ ] Run Python pipeline on 2016 dataset
- [ ] Verify reproduction of published Tables 2, 3, 5, B.2

In [ ]:
import pandas as pd
df = pd.read_stata('dropbox/data/data2025/all2025.dta', convert_categoricals=False)
df.columns.tolist()

In [6]:
df['multiaffil'].value_counts(dropna=False)


multiaffil
0.0    1145
1.0     714
NaN      49
Name: count, dtype: int64

In [ ]:
df['multiaffil'].mean()  # proportion with co-affiliations

np.float32(0.38407746)

In [4]:
df.groupby('country')['multiaffil'].agg(['mean', 'count'])

,mean,count
country,,
de,0.344891,548
en,0.395604,364
ja,0.402323,947


## Data Prep

### Building the seniority levels

In [10]:
# Drop "not employed" (code 1) — matches 2016 script line 90
df = df[df['position'] != 1].copy()

# Build 3-level seniority
#   Junior:  2 (PhD student), 3 (Post-doc), 5 (Research fellow)
#   Mid:     6 (Senior research fellow), 8 (Assistant professor)
#   Senior:  9 (Associate professor), 10 (Full professor), 12 (Emeritus)
#
# 2016 reference: lines 49-64 of prep_SPP_2016survey.do
#   junior   <- position < 5  → codes 2, 3 (and 4 if present)
#   junior   <- position == 5
#   mid      <- position in {6, 7, 8}
#   senior   <- position in {9, 10, 12}
#   (code 11 "Other" → missing → dropped; not present in 2025)
#
# Deviation from 2016: code 12 ("Emeritus" in 2025) grouped with Senior.
# The 2016 script assigned code 12 to Senior, and in 2025 code 12 is
# "Emeritus" — which we're treating as Senior (consistent with paper's
# intent of grouping full/associate profs together).

import numpy as np

conditions = [
    df['position'].isin([2, 3, 4, 5]),   # Junior
    df['position'].isin([6, 7, 8]),      # Mid
    df['position'].isin([9, 10, 12]),    # Senior
]
df['seniority'] = np.select(conditions, [1, 2, 3], default=np.nan)

# Dummies for regressions
df['junior'] = (df['seniority'] == 1).astype(int)
df['mid']    = (df['seniority'] == 2).astype(int)
df['senior'] = (df['seniority'] == 3).astype(int)

In [13]:
# Current sample state after seniority recode
print("Total rows:", len(df))
print("\nBy country:")
print(df['country'].value_counts())
print("\nSeniority × country:")
print(pd.crosstab(df['seniority'].map({1:'junior', 2:'mid', 3:'senior'}), df['country'], dropna=False, margins=True))
print("\nCo-affiliation × seniority:")
print(pd.crosstab(df['multiaffil'], df['seniority'].map({1:'junior', 2:'mid', 3:'senior'}), dropna=False, margins=True))

Total rows: 1864

By country:
country
ja    938
de    551
en    375
Name: count, dtype: int64

Seniority × country:
country     de   en   ja   All
seniority                     
junior      97   36   57   190
mid        165   69  141   375
senior     285  263  733  1281
NaN          4    7    7    18
All        551  375  938  1864

Co-affiliation × seniority:
seniority   junior  mid  senior  NaN   All
multiaffil                                
0.0            135  236     744    4  1119
1.0             51  134     512    1   698
NaN              4    5      25   13    47
All            190  375    1281   18  1864


### Binary + 4-level categorical versions of the 12 B12 motivation items

In [14]:
mot_cols = [c for c in df.columns if 'mot' in c.lower() or c.lower().startswith('b12') or 'affmot' in c.lower()]
print(mot_cols)

['af_mot_prestige', 'af_mot_network', 'af_mot_teaching', 'af_mot_income', 'af_mot_funding', 'af_mot_resources', 'af_mot_students', 'af_mot_tech', 'af_mot_ownjob', 'af_mot_jobopport', 'af_mot_exchange', 'af_mot_familyother']


In [ ]:
mot_cols = ['af_mot_prestige', 'af_mot_network', 'af_mot_teaching',
            'af_mot_income', 'af_mot_funding', 'af_mot_resources',
            'af_mot_students', 'af_mot_tech', 'af_mot_ownjob',
            'af_mot_jobopport', 'af_mot_exchange', 'af_mot_familyother']

# Binary version: 1 if "Very important" (=4), else 0.
# For non-co-affiliated respondents, force to 0 (they didn't see the question).
# 2016 ref: lines 76-82 of prep_SPP_2016survey.do
for c in mot_cols:
    df[f'bin_{c}'] = (df[c] == 4).astype(int)
    df.loc[df['multiaffil'] != 1, f'bin_{c}'] = 0

# Categorical version: 4-level ordinal with "Not applicable" (=0) folded into
# "Not at all important" (=1). This is what the factor analysis uses.
# 2016 ref: lines 84-87 of prep_SPP_2016survey.do
for c in mot_cols:
    df[f'cat_{c}'] = df[c].replace(0, 1)
    # leaves NaN as NaN — will be imputed later by seniority × country × field


af_mot_prestige:
af_mot_prestige
0.0      39
1.0      63
2.0     183
3.0     246
4.0     139
NaN    1194
Name: count, dtype: int64

af_mot_network:
af_mot_network
0.0      33
1.0      36
2.0      89
3.0     291
4.0     227
NaN    1188
Name: count, dtype: int64

af_mot_teaching:
af_mot_teaching
0.0     165
1.0     129
2.0     141
3.0     141
4.0      86
NaN    1202
Name: count, dtype: int64

af_mot_income:
af_mot_income
0.0     201
1.0     106
2.0     161
3.0     127
4.0      78
NaN    1191
Name: count, dtype: int64

af_mot_funding:
af_mot_funding
0.0     194
1.0      90
2.0     142
3.0     163
4.0      79
NaN    1196
Name: count, dtype: int64

af_mot_resources:
af_mot_resources
0.0     128
1.0     104
2.0     178
3.0     164
4.0      92
NaN    1198
Name: count, dtype: int64

af_mot_students:
af_mot_students
0.0     231
1.0     137
2.0     147
3.0     104
4.0      50
NaN    1195
Name: count, dtype: int64

af_mot_tech:
af_mot_tech
0.0     163
1.0     114
2.0     148
3.0     147
4.0     